# Pregunta 2 — Mapeo de inundaciones con Sentinel-1 y U-Net
**Redes Neuronales y Aprendizaje Profundo — UNI FIIS, 2026-II · Examen parcial**

Este notebook resuelve la Pregunta 2 de punta a punta:

| Sección | Qué hace | Entregable |
|---|---|---|
| 0–2 | Configuración, descarga de Sen1Floods11 y carga en memoria | — |
| 3–5 | U-Net propia, pérdidas (BCE, Dice, combinada, BCE con peso en bordes) y métricas | 1 |
| 6–7 | Entrenamiento y estudio de ablación (dorado vs. dorado+débil; pérdida; arquitectura) | 2 |
| 8 | Evaluación en distribución (IoU, F1, precisión, recall, matriz de confusión) + línea base Otsu | 3 |
| 9 | Galería de errores característicos del SAR | 5 |
| 10–12 | Región de prueba **Lima, 17-mar-2017** (Sentinel-1 vía Earth Engine + delineación EMSN038) | 4 |
| 13 | Tabla resumen para el informe | — |

**Base teórica:** Ronneberger et al. (2015) y la Clase 9 (U-Net). La arquitectura está escrita desde cero en PyTorch, sin bibliotecas de segmentación.

> ⚙️ En Colab: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**. Ejecute las celdas en orden. Todos los resultados (pesos, tablas, figuras) se guardan en Google Drive, así que si Colab se desconecta, las corridas ya terminadas no se repiten.

## 0. Configuración
Edite **solo** esta celda. `QUICK_RUN = True` sirve para comprobar en ~5 min que todo corre; para los resultados del informe, déjelo en `False`.

In [ ]:
# ======================= PARÁMETROS DEL PROYECTO =======================
# (cada parámetro puede sobrescribirse con una variable de entorno: así lo usa run_all.py)
import os
QUICK_RUN   = os.environ.get("P2_QUICK", "0") == "1"      # True = prueba rápida (pocos pasos)
EE_PROJECT  = os.environ.get("P2_EE_PROJECT", "")         # ID del proyecto de Google Earth Engine, p. ej. "flood-lima-uni"
N_WEAK      = int(os.environ.get("P2_N_WEAK", "1500"))    # nº de chips con etiqueta débil a descargar (de 4,385). Ver §1
DRIVE_DIR   = os.environ.get("P2_OUT",  "/content/drive/MyDrive/P2_inundaciones")   # resultados persistentes
DATA_DIR    = os.environ.get("P2_DATA", "/content/data")                            # datos (disco local)
SEED        = 42
# =======================================================================

In [ ]:
import os, sys, json, time, math, random, glob, subprocess, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run("pip install -q rasterio geopandas earthengine-api", shell=True)
    if DRIVE_DIR.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")

import torch, torch.nn as nn, torch.nn.functional as F
import rasterio
from rasterio.transform import Affine

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "SIN GPU")

CKPT_DIR, FIG_DIR, RES_DIR = (os.path.join(DRIVE_DIR, d) for d in ("checkpoints", "figuras", "resultados"))
for d in (DATA_DIR, CKPT_DIR, FIG_DIR, RES_DIR): os.makedirs(d, exist_ok=True)
RESULTS_JSON = os.path.join(RES_DIR, "experimentos.json")

## 1. Descarga de Sen1Floods11

El dataset está en el bucket público `gs://sen1floods11` (versión `v1.1`); se descarga con el cliente oficial de Google Cloud Storage en modo anónimo. Descargamos:

| Carpeta | Contenido | Uso |
|---|---|---|
| `HandLabeled/S1Hand` + `LabelHand` | 446 chips SAR (VV, VH en dB) + máscara dorada (−1 sin datos, 0 no agua, 1 agua) | entrenamiento, validación, prueba |
| `HandLabeled/S1OtsuLabelHand` | Umbral de Otsu sobre VH en los mismos 446 chips | **línea base clásica** |
| `WeaklyLabeled/S1Weak` + `S1OtsuLabelWeak` | chips SAR + etiqueta débil (Otsu) | ablación (a) |
| `splits/flood_handlabeled` | particiones oficiales train / valid / test / bolivia | — |

**¿Por qué solo `N_WEAK` chips débiles?** Los 4,385 pesan ~9 GB. Con el presupuesto de cómputo fijo que usamos en la ablación (mismo número de pasos de gradiente para todas las corridas), la red no alcanza a ver todos de todas formas. 1,500 chips son ~6 veces el conjunto dorado de entrenamiento, suficiente para medir el efecto de "muchas etiquetas ruidosas". Excluimos los chips débiles de **Bolivia** para que Bolivia siga siendo una región nunca vista.

In [ ]:
# Descarga directa desde el bucket público con el cliente de Google Cloud Storage (anónimo, sin credenciales).
# Funciona igual en Colab y en una máquina local, y descarga en paralelo.
from google.cloud import storage
from concurrent.futures import ThreadPoolExecutor
import logging; logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)   # silencia avisos inofensivos

GCS = storage.Client.create_anonymous_client(); BUCKET = GCS.bucket("sen1floods11")
ROOT = "v1.1"; HL = f"{ROOT}/data/flood_events/HandLabeled"; WL = f"{ROOT}/data/flood_events/WeaklyLabeled"

def n_files(d): return len(glob.glob(os.path.join(d, "*.tif")))

def _dl(job):
    src, dst = job
    if os.path.exists(dst) and os.path.getsize(dst) > 0: return None
    try:
        BUCKET.blob(src).download_to_filename(dst); return None
    except Exception as e:
        if os.path.exists(dst): os.remove(dst)
        return f"{src}: {e}"

def download(jobs, workers=32):
    for _, dst in jobs: os.makedirs(os.path.dirname(dst), exist_ok=True)
    t0 = time.time()
    with ThreadPoolExecutor(workers) as ex: errs = [e for e in ex.map(_dl, jobs) if e]
    print(f"  {len(jobs)} archivos en {time.time()-t0:.0f}s | errores: {len(errs)}", *errs[:3], sep="\n")

def list_tifs(prefix):
    return sorted(b.name for b in GCS.list_blobs("sen1floods11", prefix=prefix) if b.name.endswith(".tif"))

# 1) particiones oficiales
download([(f"{ROOT}/splits/flood_handlabeled/flood_{s}_data.csv", f"{DATA_DIR}/splits/flood_{s}_data.csv")
          for s in ["train", "valid", "test", "bolivia"]])

# 2) chips dorados + Otsu de los mismos chips
for sub in ["S1Hand", "LabelHand", "S1OtsuLabelHand"]:
    if n_files(f"{DATA_DIR}/{sub}") < 446:
        download([(n, f"{DATA_DIR}/{sub}/{os.path.basename(n)}") for n in list_tifs(f"{HL}/{sub}/")])
    print(f"{sub:16s}: {n_files(f'{DATA_DIR}/{sub}')} archivos")

In [ ]:
# 3) subconjunto aleatorio (reproducible) de chips débiles, sin Bolivia
if n_files(f"{DATA_DIR}/S1Weak") < N_WEAK:
    names = [n for n in list_tifs(f"{WL}/S1Weak/") if not os.path.basename(n).startswith("Bolivia")]
    pick = sorted(np.random.default_rng(SEED).choice(names, size=min(N_WEAK, len(names)), replace=False))
    print(f"Chips débiles disponibles (sin Bolivia): {len(names)} → descargamos {len(pick)}")
    jobs = []
    for n in pick:
        b = os.path.basename(n); lb = b.replace("_S1Weak.tif", "_S1OtsuLabelWeak.tif")
        jobs += [(n, f"{DATA_DIR}/S1Weak/{b}"), (f"{WL}/S1OtsuLabelWeak/{lb}", f"{DATA_DIR}/S1OtsuLabelWeak/{lb}")]
    download(jobs)
print("S1Weak:", n_files(f"{DATA_DIR}/S1Weak"), "| S1OtsuLabelWeak:", n_files(f"{DATA_DIR}/S1OtsuLabelWeak"))

## 2. Carga en memoria y normalización

**Preprocesamiento (igual que los autores del dataset):** los valores de retrodispersión en dB se recortan a $[-50, 1]$ y se llevan a $[0,1]$. Los guardamos como `uint8` (0–255): así 2,000 chips caben en ~1 GB de RAM y el entrenamiento no lee disco.

- Píxeles **NaN** del SAR → se marcan como *sin datos* en la etiqueta (−1), igual que los −1 de la etiqueta dorada. Esos píxeles **no cuentan** ni en la pérdida ni en las métricas.
- Luego estandarizamos por banda $(x-\mu)/\sigma$ con $\mu, \sigma$ calculados **solo** con los chips dorados de entrenamiento (evita fuga de información de validación/prueba).

In [ ]:
DB_MIN, DB_MAX = -50.0, 1.0

def db_to_u8(x_db):
    # x_db: (2,H,W) float en dB → uint8 (2,H,W) y máscara de NaN (H,W)
    nan = np.isnan(x_db).any(0)
    x = np.clip(np.nan_to_num(x_db, nan=DB_MIN), DB_MIN, DB_MAX)
    return np.round((x - DB_MIN) / (DB_MAX - DB_MIN) * 255).astype(np.uint8), nan

def read_tif(path):
    with rasterio.open(path) as r: return r.read()

def load_split(csv_name):
    df = pd.read_csv(f"{DATA_DIR}/splits/{csv_name}", header=None, names=["s1", "lbl"])
    return [(r.s1.strip(), r.lbl.strip()) for r in df.itertuples()]

def load_pairs(pairs, s1_dir, lbl_dir):
    X, Y, names = [], [], []
    for s1, lb in pairs:
        p1, p2 = os.path.join(DATA_DIR, s1_dir, s1), os.path.join(DATA_DIR, lbl_dir, lb)
        if not (os.path.exists(p1) and os.path.exists(p2)): continue
        x8, nan = db_to_u8(read_tif(p1)[:2].astype(np.float32))
        y = read_tif(p2)[0].astype(np.int8)
        y[(y != 0) & (y != 1)] = -1          # cualquier otro valor = sin datos
        y[nan] = -1
        X.append(x8); Y.append(y); names.append(s1.replace("_S1Hand.tif", "").replace("_S1Weak.tif", ""))
    if not X: return np.zeros((0, 2, 512, 512), np.uint8), np.zeros((0, 512, 512), np.int8), []
    return np.stack(X), np.stack(Y), names

split_pairs = {s: load_split(f"flood_{s}_data.csv") for s in ["train", "valid", "test"]}
bol_csv = f"{DATA_DIR}/splits/flood_bolivia_data.csv"
if os.path.exists(bol_csv):
    split_pairs["bolivia"] = load_split("flood_bolivia_data.csv")
else:  # respaldo: chips de Bolivia que no estén en las otras particiones
    used = {a for v in split_pairs.values() for a, _ in v}
    split_pairs["bolivia"] = [(os.path.basename(p), os.path.basename(p).replace("S1Hand", "LabelHand"))
                              for p in sorted(glob.glob(f"{DATA_DIR}/S1Hand/Bolivia_*")) if os.path.basename(p) not in used]

GOLD = {s: load_pairs(p, "S1Hand", "LabelHand") for s, p in split_pairs.items()}
OTSU = {s: load_pairs([(a, a.replace("_S1Hand", "_S1OtsuLabelHand")) for a, _ in p], "S1Hand", "S1OtsuLabelHand")
        for s, p in split_pairs.items()}

weak_pairs = [(os.path.basename(p), os.path.basename(p).replace("_S1Weak.tif", "_S1OtsuLabelWeak.tif"))
              for p in sorted(glob.glob(f"{DATA_DIR}/S1Weak/*.tif"))]
WEAK = load_pairs(weak_pairs, "S1Weak", "S1OtsuLabelWeak") if weak_pairs else (None, None, [])

for s, (X, Y, n) in GOLD.items():
    v = Y >= 0
    print(f"{s:8s}: {len(n):4d} chips | % agua (píxeles válidos) = {100*(Y[v]==1).mean():5.1f}% | eventos: {sorted({k.split('_')[0] for k in n})}")
if WEAK[0] is not None:
    v = WEAK[1] >= 0; print(f"débil   : {len(WEAK[2]):4d} chips | % agua = {100*(WEAK[1][v]==1).mean():5.1f}%")

# estadísticas de normalización: solo chips dorados de entrenamiento, solo píxeles válidos
Xtr, Ytr, _ = GOLD["train"]
vals = Xtr.astype(np.float32).transpose(1, 0, 2, 3)[:, Ytr >= 0] / 255.0
MEAN, STD = vals.mean(1, dtype=np.float64).astype(np.float32), vals.std(1, dtype=np.float64).astype(np.float32)   # float64: evita la saturación del acumulador float32 (2^24)
print("Media por banda (VV, VH):", MEAN.round(4), "| Desv.:", STD.round(4))
json.dump({"mean": MEAN.tolist(), "std": STD.tolist(), "db_range": [DB_MIN, DB_MAX]},
          open(os.path.join(RES_DIR, "normalizacion.json"), "w"))
del vals

In [ ]:
# Vistazo a 4 chips de entrenamiento: VV, VH y etiqueta dorada (gris = sin datos)
def u8_to_db(x8): return x8.astype(np.float32) / 255 * (DB_MAX - DB_MIN) + DB_MIN
LBL_CMAP = plt.matplotlib.colors.ListedColormap(["#bdbdbd", "#f2efe6", "#1f6fd1"])  # -1, 0, 1

X, Y, N = GOLD["train"]
ids = np.random.default_rng(1).choice(len(N), 4, replace=False)
fig, ax = plt.subplots(3, 4, figsize=(14, 10.5))
for j, i in enumerate(ids):
    ax[0, j].imshow(u8_to_db(X[i, 0]), cmap="gray", vmin=-25, vmax=0); ax[0, j].set_title(f"{N[i]}\nVV (dB)", fontsize=9)
    ax[1, j].imshow(u8_to_db(X[i, 1]), cmap="gray", vmin=-30, vmax=-5); ax[1, j].set_title("VH (dB)", fontsize=9)
    ax[2, j].imshow(Y[i], cmap=LBL_CMAP, vmin=-1, vmax=1, interpolation="nearest"); ax[2, j].set_title("Etiqueta dorada", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig_chips_ejemplo.png", dpi=130, bbox_inches="tight"); plt.show()

## 3. Dataset y aumento de datos

- **Recortes aleatorios de 256×256** dentro de cada chip de 512×512 (igual que los autores): más variedad por chip y lotes más grandes en la GPU.
- **Volteos y rotaciones de 90°**: el agua no tiene orientación preferida.
- **Deformación elástica** (Ronneberger et al., §3.1; Clase 9 §A7): campo de desplazamientos aleatorios en una grilla gruesa 3×3 (σ = 10 px), interpolado de forma suave y aplicado igual a imagen y máscara. Es la herramienta que el paper propone para entrenar con **pocas etiquetas**, por eso la usamos en la rama "solo dorado" de la ablación.
- No se usan ajustes de brillo/color: en SAR, la intensidad **es** la señal física (agua = baja retrodispersión).

In [ ]:
def to_tensor(x8):
    x = torch.from_numpy(x8).float() / 255.0
    return (x - torch.tensor(MEAN).view(2, 1, 1)) / torch.tensor(STD).view(2, 1, 1)

def elastic(x, y, grid=3, sigma=10.0):
    # x: (C,H,W) float, y: (H,W) float con -1/0/1. Misma deformación suave para ambos.
    C, H, W = x.shape
    disp = torch.randn(1, 2, grid, grid) * sigma
    disp = F.interpolate(disp, size=(H, W), mode="bicubic", align_corners=True)[0]
    base_y, base_x = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing="ij")
    gx = base_x + disp[0] * 2 / (W - 1); gy = base_y + disp[1] * 2 / (H - 1)
    g = torch.stack([gx, gy], -1)[None]
    x = F.grid_sample(x[None], g, mode="bilinear", padding_mode="reflection", align_corners=True)[0]
    y = F.grid_sample(y[None, None], g, mode="nearest", padding_mode="reflection", align_corners=True)[0, 0]
    return x, y

class FloodChips(torch.utils.data.Dataset):
    def __init__(self, X, Y, crop=256, augment=False, elastic_p=0.0):
        self.X, self.Y, self.crop, self.augment, self.elastic_p = X, Y, crop, augment, elastic_p
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x, y = self.X[i], self.Y[i]
        if self.crop and self.crop < y.shape[-1]:
            r, c = np.random.randint(0, y.shape[0] - self.crop + 1), np.random.randint(0, y.shape[1] - self.crop + 1)
            x, y = x[:, r:r+self.crop, c:c+self.crop], y[r:r+self.crop, c:c+self.crop]
        x, y = to_tensor(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y)).float()
        if self.augment:
            k = np.random.randint(4); x, y = torch.rot90(x, k, (1, 2)), torch.rot90(y, k, (0, 1))
            if np.random.rand() < 0.5: x, y = x.flip(2), y.flip(1)
            if np.random.rand() < self.elastic_p: x, y = elastic(x, y)
        return x, y

x, y = FloodChips(*GOLD["train"][:2], augment=True, elastic_p=1.0)[0]
print("Ejemplo de muestra:", tuple(x.shape), tuple(y.shape), "| valores de etiqueta:", torch.unique(y).tolist())

## 4. U-Net propia (Entregable 1)

Implementación desde cero siguiendo la Fig. 1 del paper, con los tres cambios "modernos" vistos en clase (§A5): *padding* = 1 (la salida tiene el mismo tamaño que la entrada, sin recortes), Batch Normalization e inicialización de He.

- **Entrada:** 2 canales (VV, VH). **Salida:** 1 canal de *logits* (agua sí/no) → sigmoide.
- `depth`: número de niveles de *max-pooling* (paper: 4). `base`: canales del primer nivel (paper: 64; usamos 32 por memoria y tiempo).
- `skip=False` elimina las conexiones de salto (el decodificador solo recibe el camino profundo) → ablación arquitectónica.
- `up="transpose"` usa *up-convolution* 2×2 como el paper; `up="bilinear"` usa interpolación + conv 1×1 (evita el efecto tablero de ajedrez).

**Nota sobre BCE vs. softmax:** con dos clases, *softmax* + entropía cruzada sobre 2 logits es matemáticamente equivalente a *sigmoide* + BCE sobre su diferencia; por eso usamos 1 canal de salida.

In [ ]:
class ConvBlock(nn.Module):
    # (Conv 3x3 → BN → ReLU) × 2 : el bloque básico de cada nivel de la U-Net
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UpBlock(nn.Module):
    # sube resolución ×2, (opcional) concatena el mapa del codificador, y aplica ConvBlock
    def __init__(self, cin, cout, skip=True, up="transpose"):
        super().__init__()
        self.skip = skip
        self.up = (nn.ConvTranspose2d(cin, cout, 2, stride=2) if up == "transpose" else
                   nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False), nn.Conv2d(cin, cout, 1)))
        self.conv = ConvBlock(2 * cout if skip else cout, cout)
    def forward(self, x, enc):
        x = self.up(x)
        if x.shape[-2:] != enc.shape[-2:]:
            x = F.interpolate(x, size=enc.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([enc, x], 1) if self.skip else x)

class FloodUNet(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=32, depth=4, skip=True, up="transpose"):
        super().__init__()
        ch = [base * 2 ** i for i in range(depth + 1)]                # p. ej. 32, 64, 128, 256, 512
        self.stem = ConvBlock(in_ch, ch[0])
        self.down = nn.ModuleList(nn.Sequential(nn.MaxPool2d(2), ConvBlock(ch[i], ch[i + 1])) for i in range(depth))
        self.ups = nn.ModuleList(UpBlock(ch[i + 1], ch[i], skip, up) for i in reversed(range(depth)))
        self.head = nn.Conv2d(ch[0], out_ch, 1)
        for m in self.modules():                                       # inicialización de He (paper §3)
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        feats = [self.stem(x)]
        for d in self.down: feats.append(d(feats[-1]))
        x = feats.pop()                                                # cuello de botella
        for up in self.ups: x = up(x, feats.pop())
        return self.head(x)[:, 0]                                      # (N,H,W) logits

for kw in [dict(), dict(skip=False), dict(depth=3)]:
    m = FloodUNet(**kw)
    print(f"{str(kw):16s} → {sum(p.numel() for p in m.parameters())/1e6:5.2f} M parámetros | "
          f"(2,2,256,256) → {tuple(m(torch.randn(2, 2, 256, 256)).shape)}")

## 5. Funciones de pérdida y métricas

Todas las pérdidas ignoran los píxeles *sin datos* (máscara $v$).

- **BCE:** $\mathcal{L}_{BCE} = -\frac{1}{|v|}\sum_{x\in v} \big[y\log p + (1-y)\log(1-p)\big]$
- **Dice (suave):** $\mathcal{L}_{Dice} = 1 - \frac{2\sum p\,y + 1}{\sum p + \sum y + 1}$ — robusta al desbalance (el agua es minoría).
- **Combinada:** $\mathcal{L}_{BCE} + \mathcal{L}_{Dice}$ (la que usa la Clase 9).
- **BCE con peso en bordes** (adaptación de la Ec. 2 del paper, ejercicio 5 de las notas de clase): $w(x) = 1 + w_0 \cdot G_\sigma * B(x)$, donde $B$ es el borde agua/no-agua y $G_\sigma$ un suavizado gaussiano. En Lima, la inundación está en franjas angostas junto a los ríos, donde casi todo es "borde".

**Métricas de la clase agua** (sobre píxeles válidos): IoU $=\frac{TP}{TP+FP+FN}$, precisión, recall, F1, y las tasas de **omisión** $=\frac{FN}{TP+FN}$ y **comisión** $=\frac{FP}{TP+FP}$ que reportan los autores del dataset. Reportamos el IoU **agregado** (todos los píxeles juntos) y el IoU **promedio por chip** (métrica de referencia de los autores).

In [ ]:
def bce_loss(logits, y, w=None):
    v = (y >= 0).float(); t = y.clamp(min=0)
    l = F.binary_cross_entropy_with_logits(logits, t, reduction="none") * v
    if w is not None: l = l * w
    return l.sum() / v.sum().clamp(min=1)

def dice_loss(logits, y, eps=1.0):
    v = (y >= 0).float(); t = y.clamp(min=0) * v; p = torch.sigmoid(logits) * v
    return 1 - (2 * (p * t).sum() + eps) / (p.sum() + t.sum() + eps)

def edge_weights(y, w0=5.0, sigma=3.0):
    # borde = píxeles donde el máximo y el mínimo de su vecindad 3x3 difieren (agua ↔ no agua)
    t = y.clamp(min=0)[:, None]
    border = (F.max_pool2d(t, 3, 1, 1) - (-F.max_pool2d(-t, 3, 1, 1))).clamp(0, 1)
    r = int(3 * sigma); k = torch.exp(-torch.arange(-r, r + 1, device=y.device).float() ** 2 / (2 * sigma ** 2)); k = k / k.max()
    b = F.conv2d(border, k.view(1, 1, 1, -1), padding=(0, r)); b = F.conv2d(b, k.view(1, 1, -1, 1), padding=(r, 0))
    b = b / b.amax(dim=(2, 3), keepdim=True).clamp(min=1e-6)
    return (1 + w0 * b)[:, 0]

LOSSES = {
    "bce":      lambda lg, y: bce_loss(lg, y),
    "dice":     lambda lg, y: dice_loss(lg, y),
    "bce+dice": lambda lg, y: bce_loss(lg, y) + dice_loss(lg, y),
    "bce_borde":lambda lg, y: bce_loss(lg, y, edge_weights(y)),
}

class WaterMetrics:
    # acumula la matriz de confusión binaria (solo píxeles válidos) y el IoU por chip
    def __init__(self): self.tp = self.fp = self.fn = self.tn = 0; self.chip_iou = []
    def update(self, pred, y):
        for p, t in zip(pred, y):
            v = t >= 0; p, t = p[v].bool(), t[v] == 1
            tp, fp, fn = (p & t).sum().item(), (p & ~t).sum().item(), (~p & t).sum().item()
            self.tp += tp; self.fp += fp; self.fn += fn; self.tn += (~p & ~t).sum().item()
            if tp + fp + fn > 0: self.chip_iou.append(tp / (tp + fp + fn))
    def summary(self):
        tp, fp, fn, tn = self.tp, self.fp, self.fn, self.tn; e = 1e-9
        prec, rec = tp / (tp + fp + e), tp / (tp + fn + e)
        return {"IoU": tp / (tp + fp + fn + e), "IoU_chip_medio": float(np.mean(self.chip_iou)) if self.chip_iou else float("nan"),
                "F1": 2 * prec * rec / (prec + rec + e), "precision": prec, "recall": rec,
                "omision": fn / (tp + fn + e), "comision": fp / (tp + fp + e), "exactitud": (tp + tn) / (tp + fp + fn + tn + e),
                "TP": tp, "FP": fp, "FN": fn, "TN": tn}

# prueba rápida de las pérdidas y métricas
lg = torch.randn(2, 64, 64); yy = torch.randint(-1, 2, (2, 64, 64)).float()
print({k: round(f(lg, yy).item(), 4) for k, f in LOSSES.items()})
m = WaterMetrics(); m.update((lg > 0).long(), yy); print({k: round(v, 3) for k, v in m.summary().items() if k[0].islower() or k[0] == "I"})

## 6. Entrenamiento

Decisión de diseño importante para que la ablación sea **justa**: todas las corridas usan el **mismo presupuesto de cómputo** (mismo número de pasos de gradiente y tamaño de lote). Así, "dorado + débil" no gana solo por haber entrenado más tiempo.

- Optimizador AdamW + *OneCycle* (como la Clase 9), precisión mixta en GPU.
- Cada `EVAL_EVERY` pasos se evalúa en **validación** (chips dorados completos de 512×512) y se guarda el mejor modelo por IoU de agua (*early stopping* implícito).
- **Mezcla dorado + débil:** se muestrea con probabilidad `gold_frac` desde los chips dorados y `1 - gold_frac` desde los débiles. Con 50/50 evitamos que las etiquetas ruidosas (6× más numerosas) "ahoguen" a las expertas.
- Cada corrida se registra en `experimentos.json` en Drive; si ya existe, se salta (útil si Colab se desconecta).

In [ ]:
STEPS      = 300 if QUICK_RUN else 3000
EVAL_EVERY = 100 if QUICK_RUN else 250
BATCH      = 16
LR         = 2e-3

def load_results():
    return json.load(open(RESULTS_JSON)) if os.path.exists(RESULTS_JSON) else {}

def build_train_loader(data, elastic_p, gold_frac=0.5):
    gX, gY, _ = GOLD["train"]
    if data == "gold" or WEAK[0] is None:
        ds = FloodChips(gX, gY, augment=True, elastic_p=elastic_p); weights = np.ones(len(ds))
    else:
        wX, wY, _ = WEAK
        ds = torch.utils.data.ConcatDataset([FloodChips(gX, gY, augment=True, elastic_p=elastic_p),
                                             FloodChips(wX, wY, augment=True, elastic_p=elastic_p)])
        weights = np.r_[np.full(len(gX), gold_frac / len(gX)), np.full(len(wX), (1 - gold_frac) / len(wX))]
    sampler = torch.utils.data.WeightedRandomSampler(weights, num_samples=STEPS * BATCH, replacement=True)
    return torch.utils.data.DataLoader(ds, batch_size=BATCH, sampler=sampler, num_workers=2 if IN_COLAB else 0,
                                       pin_memory=device.type == "cuda", drop_last=True,
                                       worker_init_fn=lambda w: np.random.seed((torch.initial_seed() + w) % 2**32))

@torch.no_grad()
def predict_chips(model, X, bs=4):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        xb = torch.stack([to_tensor(x) for x in X[i:i + bs]]).to(device)
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            out.append(torch.sigmoid(model(xb).float()).cpu())
    return torch.cat(out)

def evaluate(model, X, Y, thr=0.5):
    m = WaterMetrics(); m.update((predict_chips(model, X) > thr).long(), torch.from_numpy(Y)); return m.summary()

def make_model(cfg):
    return FloodUNet(base=cfg.get("base", 32), depth=cfg.get("depth", 4), skip=cfg.get("skip", True), up=cfg.get("up", "transpose"))

def run_experiment(name, cfg, force=False):
    res = load_results(); ck = f"{CKPT_DIR}/{name}.pt"
    if name in res and os.path.exists(ck) and not force and res[name].get("steps") == STEPS:
        print(f"[{name}] ya existe → se omite (IoU val = {res[name]['val']['IoU']:.3f})"); return res[name]
    seed_everything(SEED)
    model = make_model(cfg).to(device)
    loader = build_train_loader(cfg["data"], cfg.get("elastic_p", 0.3), cfg.get("gold_frac", 0.5))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS)
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
    loss_fn = LOSSES[cfg["loss"]]
    hist = {"step": [], "train_loss": [], "val_IoU": []}; best, run, t0 = -1, 0.0, time.time()
    gX, gY, _ = GOLD["valid"]
    for step, (x, y) in enumerate(loader, 1):
        model.train(); x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            logits = model(x)
        loss = loss_fn(logits.float(), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
        if step % EVAL_EVERY == 0 or step == STEPS:
            v = evaluate(model, gX, gY)
            hist["step"].append(step); hist["train_loss"].append(run / EVAL_EVERY); hist["val_IoU"].append(v["IoU"]); run = 0.0
            if v["IoU"] > best: best = v["IoU"]; torch.save(model.state_dict(), ck)
            print(f"[{name}] paso {step:5d}/{STEPS} | pérdida {hist['train_loss'][-1]:.4f} | IoU val {v['IoU']:.3f} | {time.time()-t0:.0f}s")
    model.load_state_dict(torch.load(ck, map_location=device))
    rec = {"cfg": cfg, "steps": STEPS, "hist": hist, "minutes": (time.time() - t0) / 60,
           "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
           "val": evaluate(model, *GOLD["valid"][:2]), "test": evaluate(model, *GOLD["test"][:2]),
           "bolivia": evaluate(model, *GOLD["bolivia"][:2]) if len(GOLD["bolivia"][2]) else None}
    res = load_results(); res[name] = rec; json.dump(res, open(RESULTS_JSON, "w"), indent=1)
    print(f"[{name}] listo en {rec['minutes']:.1f} min | IoU val {rec['val']['IoU']:.3f} | test {rec['test']['IoU']:.3f}")
    return rec

def load_model(name):
    cfg = load_results()[name]["cfg"]; m = make_model(cfg).to(device)
    m.load_state_dict(torch.load(f"{CKPT_DIR}/{name}.pt", map_location=device)); return m.eval()

## 7. Estudio de ablación (Entregable 2)

Se cambia **una sola decisión a la vez**, como en una losa de prueba. La elección del "mejor" en cada etapa se hace con el IoU de **validación**, nunca de prueba.

| Etapa | Corridas | Pregunta que responde |
|---|---|---|
| (a) Datos | **E1** solo dorado (+ deformación elástica fuerte, p = 0.5) vs **E2** dorado + débil (50/50) | ¿Pocas etiquetas buenas + aumento (tesis de U-Net) o muchas etiquetas ruidosas (hallazgo de Bonafilia et al., 2020)? |
| (b) Pérdida | Con los mejores datos: **E3** Dice, **E4** BCE + Dice, **E6** BCE con peso en bordes | ¿Qué pérdida maneja mejor el desbalance y los bordes del agua? |
| (c) Arquitectura | Con lo mejor: **E5a** sin conexiones de salto, **E5b** profundidad 3 | ¿Cuánto aportan los saltos y la profundidad? |

Tiempo estimado en T4: ~8–12 min por corrida (~1 h 15 min en total). Las corridas ya hechas se omiten.

In [ ]:
# (a) Ablación de datos — con BCE
E = {}
E["E1_dorado_bce"]       = run_experiment("E1_dorado_bce",       dict(data="gold",      loss="bce", elastic_p=0.5))
E["E2_dorado+debil_bce"] = run_experiment("E2_dorado+debil_bce", dict(data="gold+weak", loss="bce", elastic_p=0.3, gold_frac=0.5))
best_a = max(["E1_dorado_bce", "E2_dorado+debil_bce"], key=lambda k: E[k]["val"]["IoU"])
DATA_BEST = E[best_a]["cfg"]
print("Mejores datos (por IoU de validación):", best_a)

In [ ]:
# (b) Ablación de pérdida — con los mejores datos
base_cfg = {k: v for k, v in DATA_BEST.items() if k != "loss"}
E["E3_dice"]      = run_experiment("E3_dice",      dict(base_cfg, loss="dice"))
E["E4_bce+dice"]  = run_experiment("E4_bce+dice",  dict(base_cfg, loss="bce+dice"))
E["E6_bce_borde"] = run_experiment("E6_bce_borde", dict(base_cfg, loss="bce_borde"))
cands = [best_a, "E3_dice", "E4_bce+dice", "E6_bce_borde"]
best_b = max(cands, key=lambda k: E[k]["val"]["IoU"])
print("Mejor configuración datos+pérdida:", best_b)

In [ ]:
# (c) Ablación de arquitectura — sobre la mejor configuración
best_cfg = E[best_b]["cfg"]
E["E5a_sin_saltos"] = run_experiment("E5a_sin_saltos", dict(best_cfg, skip=False))
E["E5b_depth3"]     = run_experiment("E5b_depth3",     dict(best_cfg, depth=3))
BEST = max(E, key=lambda k: E[k]["val"]["IoU"])
print("MODELO FINAL (mejor IoU de validación):", BEST)
json.dump({"best": BEST}, open(os.path.join(RES_DIR, "mejor_modelo.json"), "w"))

In [ ]:
# Curvas de aprendizaje de todas las corridas
res = load_results()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k, r in res.items():
    ax[0].plot(r["hist"]["step"], r["hist"]["train_loss"], label=k); ax[1].plot(r["hist"]["step"], r["hist"]["val_IoU"], marker="o", ms=3, label=k)
ax[0].set(title="Pérdida de entrenamiento", xlabel="paso"); ax[1].set(title="IoU de agua en validación", xlabel="paso")
ax[1].legend(fontsize=8); [a.grid(alpha=.3) for a in ax]
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig_curvas.png", dpi=130, bbox_inches="tight"); plt.show()

rows = []
for k, r in res.items():
    c = r["cfg"]
    rows.append({"corrida": k, "datos": c["data"], "pérdida": c["loss"], "saltos": c.get("skip", True), "prof.": c.get("depth", 4),
                 "params (M)": round(r["params_M"], 2), "IoU val": r["val"]["IoU"], "IoU test": r["test"]["IoU"],
                 "F1 test": r["test"]["F1"], "IoU Bolivia": (r["bolivia"] or {}).get("IoU", np.nan), "min": round(r["minutes"], 1)})
tabla_ablacion = pd.DataFrame(rows).set_index("corrida")
tabla_ablacion.to_csv(f"{RES_DIR}/tabla_ablacion.csv")
tabla_ablacion.style.format(precision=3).background_gradient(subset=["IoU val", "IoU test"], cmap="Greens")

## 8. Evaluación en distribución (Entregable 3)

Modelo final sobre la **partición de prueba oficial** (chips dorados de eventos vistos en entrenamiento) y, como referencia, sobre **Bolivia** (evento nunca visto). Se compara con la **línea base clásica**: umbral de Otsu sobre VH (la misma técnica que genera las etiquetas débiles).

In [ ]:
model = load_model(BEST)
def otsu_metrics(split):
    m = WaterMetrics(); oy = OTSU[split][1]
    m.update(torch.from_numpy((oy == 1).astype(np.int64)), torch.from_numpy(GOLD[split][1])); return m.summary()

filas = {}
for split in ["test", "bolivia"]:
    if len(GOLD[split][2]) == 0: continue
    filas[f"U-Net ({split})"] = evaluate(model, *GOLD[split][:2])
    filas[f"Otsu VH ({split})"] = otsu_metrics(split)
tabla_eval = pd.DataFrame(filas).T[["IoU", "IoU_chip_medio", "F1", "precision", "recall", "omision", "comision"]]
tabla_eval.to_csv(f"{RES_DIR}/tabla_evaluacion.csv"); print(tabla_eval.round(3))

# Matriz de confusión (prueba en distribución), en conteos y normalizada por fila
s = filas["U-Net (test)"]; cm = np.array([[s["TN"], s["FP"]], [s["FN"], s["TP"]]], dtype=float)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, M, t, fmt in [(ax[0], cm / 1e6, "Conteos (millones de píxeles)", "{:.2f}"), (ax[1], cm / cm.sum(1, keepdims=True), "Normalizada por fila (= recall por clase)", "{:.3f}")]:
    a.imshow(M, cmap="Blues"); a.set_title(t, fontsize=10)
    a.set_xticks([0, 1], ["pred: no agua", "pred: agua"]); a.set_yticks([0, 1], ["real: no agua", "real: agua"])
    for i in range(2):
        for j in range(2): a.text(j, i, fmt.format(M[i, j]), ha="center", va="center", color="white" if M[i, j] > M.max() / 2 else "black")
plt.suptitle(f"Matriz de confusión — {BEST} — prueba en distribución"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig_matriz_confusion.png", dpi=130, bbox_inches="tight"); plt.show()

## 9. Galería de errores (Entregable 5)

Mapa de error por píxel: **gris oscuro** = acierto agua (TP), crema = acierto no-agua (TN), **rojo** = falso positivo (FP, el modelo ve agua donde no hay), **azul** = falso negativo (FN, agua no detectada), gris claro = sin datos.

Se muestran los chips de prueba con **más FP** y con **más FN**. Úselos para identificar y explicar en el informe al menos tres errores típicos del SAR:
1. **Sombra de radar / superficies lisas** (pistas, arena seca, pendientes opuestas al satélite): baja retrodispersión → parecen agua → **FP**.
2. **Doble rebote** en zonas urbanas o con vegetación inundada: la señal rebota en el agua y luego en la pared/tronco y vuelve **brillante** → agua no detectada → **FN**.
3. **Suelo húmedo vs. agua abierta**: el suelo saturado también oscurece la señal → bordes de inundación sobreestimados → **FP** en bordes.
4. **Viento / oleaje**: el agua rugosa retrodispersa más → partes de ríos y lagos no detectadas → **FN**.

In [ ]:
ERR_CMAP = plt.matplotlib.colors.ListedColormap(["#d9d9d9", "#f7f4ec", "#3a3a3a", "#d7301f", "#2c7fb8"])  # nodata, TN, TP, FP, FN
def error_map(pred, y):
    e = np.ones_like(y, dtype=np.int8)                  # 1 = TN
    e[(pred == 1) & (y == 1)] = 2; e[(pred == 1) & (y == 0)] = 3; e[(pred == 0) & (y == 1)] = 4; e[y < 0] = 0
    return e

def galeria(split, idx, titulo, fname):
    X, Y, N = GOLD[split]; P = (predict_chips(model, X[idx]) > 0.5).numpy()
    fig, ax = plt.subplots(len(idx), 4, figsize=(13, 3.3 * len(idx)))
    for r, (i, p) in enumerate(zip(idx, P)):
        ax[r, 0].imshow(u8_to_db(X[i, 0]), cmap="gray", vmin=-25, vmax=0); ax[r, 0].set_title(f"{N[i]} · VV", fontsize=9)
        ax[r, 1].imshow(u8_to_db(X[i, 1]), cmap="gray", vmin=-30, vmax=-5); ax[r, 1].set_title("VH", fontsize=9)
        ax[r, 2].imshow(Y[i], cmap=LBL_CMAP, vmin=-1, vmax=1, interpolation="nearest"); ax[r, 2].set_title("Etiqueta dorada", fontsize=9)
        e = error_map(p, Y[i]); ax[r, 3].imshow(e, cmap=ERR_CMAP, vmin=0, vmax=4, interpolation="nearest")
        ax[r, 3].set_title(f"Error  FP={(e==3).sum()}  FN={(e==4).sum()}", fontsize=9)
    for a in ax.ravel(): a.axis("off")
    plt.suptitle(titulo); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/{fname}", dpi=120, bbox_inches="tight"); plt.show()

X, Y, N = GOLD["test"]; P = (predict_chips(model, X) > 0.5).numpy()
fp = np.array([((p == 1) & (y == 0)).sum() for p, y in zip(P, Y)]); fn = np.array([((p == 0) & (y == 1)).sum() for p, y in zip(P, Y)])
galeria("test", list(np.argsort(-fp)[:4]), "Chips de prueba con MÁS falsos positivos", "fig_errores_FP.png")
galeria("test", list(np.argsort(-fn)[:4]), "Chips de prueba con MÁS falsos negativos", "fig_errores_FN.png")

## 10. Región de prueba Perú — Lima, 17 de marzo de 2017 (Entregable 4)

**Fuente SAR.** En ASF Vertex se encontraron tres escenas Sentinel-1 GRD IW (VV+VH) sobre Lima en marzo de 2017:

| Fecha (UTC) | Escena | Uso |
|---|---|---|
| **17-mar-2017 10:38** | S1B, órbita relativa 171, descendente (…040D) | **escena de prueba**: mismo día de los desbordes de Huachipa/Carapongo y 1 día antes de la imagen WorldView-2 (18-mar) usada por EMSN038 |
| 29-mar-2017 10:38 | misma órbita (…1AB7) | 12 días después → muestra la **recesión** del agua (hipótesis de desajuste temporal) |
| 10-mar-2017 10:46 | otra pasada (…2D06) | antes del evento, geometría distinta → solo referencia visual |

Se obtienen desde **Google Earth Engine** (`COPERNICUS/S1_GRD`), que entrega las mismas escenas ya calibradas, con corrección de terreno y en dB — el mismo procesamiento con el que se construyó Sen1Floods11. Se remuestrean a la **misma grilla que Sen1Floods11**: EPSG:4326, píxel de 8.983×10⁻⁵ ° (≈ 10 m).

**Etiquetas de referencia.** Polígonos de inundación de la activación **EMSN038** de Copernicus EMS, rasterizados sobre esa grilla. ⚠️ Provienen de imágenes ópticas de mayor resolución y no de la misma fecha/criterio que las etiquetas doradas: son una **supervisión débil propia**, no un *ground truth* perfecto.

**Máscaras de evaluación.** Solo se evalúan píxeles (i) dentro del área de interés de EMSN038, (ii) en tierra firme, (iii) que no son agua permanente según JRC Global Surface Water (ocurrencia ≥ 50 %): EMSN038 delimita la **inundación**, no el cauce permanente ni el mar.

**Antes de ejecutar:** (1) escriba su `EE_PROJECT` en la §0; (2) suba el zip (o los archivos) del producto EMSN038 a la carpeta de Drive `P2_inundaciones/EMSN038/`.

In [ ]:
import ee
assert EE_PROJECT, "Escriba el ID de su proyecto de Earth Engine en EE_PROJECT (sección 0)."
if IN_COLAB: ee.Authenticate()          # en Colab abre una ventana para autorizar; fuera de Colab, ejecutar antes `earthengine authenticate`
ee.Initialize(project=EE_PROJECT)

PIX = 8.983152841195215e-05             # tamaño de píxel de Sen1Floods11 (grados ≈ 10 m)
AOI = dict(west=-77.20, south=-12.10, east=-76.70, north=-11.74)   # Chillón + Rímac, de la costa a Chosica
W = int(round((AOI["east"] - AOI["west"]) / PIX)); H = int(round((AOI["north"] - AOI["south"]) / PIX))
LIMA_TF = Affine(PIX, 0, AOI["west"], 0, -PIX, AOI["north"])
aoi_geom = ee.Geometry.Rectangle([AOI["west"], AOI["south"], AOI["east"], AOI["north"]])
print(f"Grilla de Lima: {W} × {H} píxeles ({W*PIX*111:.0f} × {H*PIX*111:.0f} km aprox.)")

def s1_for_date(day, orbit=None):
    col = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(aoi_geom)
           .filterDate(day, ee.Date(day).advance(1, "day"))
           .filter(ee.Filter.eq("instrumentMode", "IW"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH")))
    if orbit: col = col.filter(ee.Filter.eq("relativeOrbitNumber_start", orbit))
    print(day, "→ escenas en Earth Engine:", col.aggregate_array("system:index").getInfo())
    return col.select(["VV", "VH"]).mosaic().unmask(-9999)

def fetch(image, bands, tile=1024, dtype=np.float32):
    # descarga la imagen en la grilla de Lima, por bloques (límite de tamaño por petición de Earth Engine)
    out = np.zeros((len(bands), H, W), dtype)
    for r0 in range(0, H, tile):
        for c0 in range(0, W, tile):
            h, w = min(tile, H - r0), min(tile, W - c0)
            req = {"expression": image.select(bands), "fileFormat": "NUMPY_NDARRAY",
                   "grid": {"dimensions": {"width": w, "height": h}, "crsCode": "EPSG:4326",
                            "affineTransform": {"scaleX": PIX, "shearX": 0, "translateX": AOI["west"] + c0 * PIX,
                                                "shearY": 0, "scaleY": -PIX, "translateY": AOI["north"] - r0 * PIX}}}
            arr = ee.data.computePixels(req)
            out[:, r0:r0 + h, c0:c0 + w] = np.stack([arr[b] for b in bands]).astype(dtype)
    return out

def save_geotiff(path, arr, dtype=None):
    arr = arr if arr.ndim == 3 else arr[None]
    with rasterio.open(path, "w", driver="GTiff", height=H, width=W, count=arr.shape[0], dtype=dtype or arr.dtype,
                       crs="EPSG:4326", transform=LIMA_TF, compress="deflate") as dst: dst.write(arr)

LIMA_DIR = os.path.join(DRIVE_DIR, "lima"); os.makedirs(LIMA_DIR, exist_ok=True)
LIMA = {}
for day in ["2017-03-17", "2017-03-29", "2017-03-10"]:
    f = f"{LIMA_DIR}/S1_{day}.tif"
    if not os.path.exists(f):
        a = fetch(s1_for_date(day), ["VV", "VH"]); a[a <= -9000] = np.nan; save_geotiff(f, a)
    LIMA[day] = read_tif(f)
    print(day, "| % píxeles válidos:", round(100 * (~np.isnan(LIMA[day]).any(0)).mean(), 1))

# Máscaras auxiliares: agua permanente (JRC) y tierra firme (límites de país, LSIB)
f = f"{LIMA_DIR}/aux_jrc_tierra.tif"
if not os.path.exists(f):
    jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").unmask(0).rename("jrc")
    land = ee.Image(0).paint(ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq("country_na", "Peru")), 1).rename("land")
    save_geotiff(f, fetch(jrc.addBands(land), ["jrc", "land"]))
JRC_OCC, LAND = read_tif(f)

### 10.1 Etiquetas EMSN038 → ráster

La celda **descarga** el paquete vectorial oficial de la activación (`EMSN038_Geodatabase.zip`, geodatabase ESRI) si aún no está en Drive, lo descomprime y lista todas sus capas. Selecciona automáticamente:
- **Inundación:** la capa `DamAssess_A`, que contiene la extensión de inundación detectada por análisis de objetos (OBIA) sobre la imagen óptica WorldView-2, restringida a lo **observado** a lo largo de los ríos (`source = "river"`, 660 polígonos, 6.34 km²). Se excluye la extensión extrapolada hasta la cota máxima (`source = "maximun height"`); con `GT_MODE = "todo"` se puede incluir como análisis de sensibilidad.
- **Área de interés:** la capa `EMSN038_AOI`.

In [ ]:
import geopandas as gpd, pyogrio, urllib.request, shapely
from rasterio.features import rasterize

EMSN_DIR = os.path.join(DRIVE_DIR, "EMSN038"); os.makedirs(EMSN_DIR, exist_ok=True)
EMSN_URL = "https://mapping.emergency.copernicus.eu/download/EMSN038/EMSN038_Geodatabase.zip"
if not glob.glob(f"{EMSN_DIR}/**/*.zip", recursive=True) and not glob.glob(f"{EMSN_DIR}/**/*.gdb", recursive=True):
    print("Descargando", EMSN_URL)
    req = urllib.request.Request(EMSN_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(f"{EMSN_DIR}/EMSN038_Geodatabase.zip", "wb") as f: f.write(r.read())
for z in glob.glob(f"{EMSN_DIR}/**/*.zip", recursive=True):
    out = z[:-4]
    if not os.path.exists(out): zipfile.ZipFile(z).extractall(out)
    for z2 in glob.glob(f"{out}/**/*.zip", recursive=True):          # zips anidados
        if not os.path.exists(z2[:-4]): zipfile.ZipFile(z2).extractall(z2[:-4])

# todas las capas: archivos sueltos (shp/geojson/gpkg/kml) + capas dentro de geodatabases (.gdb)
fuentes = [(p, None) for ext in ("shp", "geojson", "json", "gpkg", "kml") for p in glob.glob(f"{EMSN_DIR}/**/*.{ext}", recursive=True)]
for gdb in glob.glob(f"{EMSN_DIR}/**/*.gdb", recursive=True):
    fuentes += [(gdb, lyr) for lyr, _ in pyogrio.list_layers(gdb)]
capas = []
for p, lyr in sorted(fuentes, key=lambda t: (t[0], t[1] or "")):
    nombre = (os.path.basename(p) + (f"::{lyr}" if lyr else ""))
    try: g = gpd.read_file(p, layer=lyr) if lyr else gpd.read_file(p)
    except Exception as e: print("No se pudo leer", nombre, e); continue
    if len(g) == 0 or g.geometry is None: continue
    capas.append((nombre, g)); print(f"{nombre:85s} | {len(g):5d} entidades | {g.geom_type.unique().tolist()} | CRS {g.crs}")

# Estructura real del paquete EMSN038 (producto "Risk & Recovery"): la extensión de inundación óptica (OBIA)
# está en la capa DamAssess_A; su campo `source` distingue lo OBSERVADO a lo largo de los ríos ("river")
# de la extensión extrapolada hasta la cota máxima del agua ("maximun height"). El área de interés es EMSN038_AOI.
GT_MODE = "river"          # "river" = inundación observada (principal) | "todo" = observada + extrapolada
cap = dict(capas)
def _capa(sufijo):
    k = [n for n in cap if n.lower().endswith("::" + sufijo.lower())]
    return cap[k[0]] if k else None
dam, aoi = _capa("DamAssess_A"), _capa("EMSN038_AOI")
if dam is not None:
    sel = dam[dam["source"] == "river"] if GT_MODE == "river" else dam
else:   # respaldo para paquetes con nomenclatura estándar de Copernicus EMS (observedEvent / areaOfInterest)
    cand = [g for n, g in capas if "observedevent" in n.lower() and not any(k in n.lower() for k in ("landslide", "damage", "grading"))]
    sel = pd.concat(cand) if cand else None
    cand = [g for n, g in capas if "areaofinterest" in n.lower()]
    aoi = pd.concat([g.to_crs(4326) for g in cand]) if cand else None
assert sel is not None, "No se encontró la capa de inundación: revise la lista de capas impresa arriba."
flood_gdf = sel.to_crs(4326)[["geometry"]].copy(); flood_gdf["geometry"] = shapely.force_2d(flood_gdf.geometry.values)
aoi_gdf = None
if aoi is not None:
    aoi_gdf = aoi.to_crs(4326)[["geometry"]].copy(); aoi_gdf["geometry"] = shapely.force_2d(aoi_gdf.geometry.values)
print(f"Etiqueta EMSN038 ({GT_MODE}): {len(flood_gdf)} polígonos | área = {sel.to_crs(32718).area.sum()/1e6:.2f} km²")

def burn(gdf):
    return rasterize(((geom, 1) for geom in gdf.geometry if geom is not None and not geom.is_empty),
                     out_shape=(H, W), transform=LIMA_TF, fill=0, dtype="uint8")
FLOOD = burn(flood_gdf)
IN_AOI = burn(aoi_gdf) if aoi_gdf is not None else np.ones((H, W), np.uint8)
if aoi_gdf is None: print("⚠️ Sin capa de área de interés: se evalúa en toda la grilla (documentarlo en el informe).")

VALID = (IN_AOI == 1) & (LAND == 1) & (JRC_OCC < 50) & ~np.isnan(LIMA["2017-03-17"]).any(0)
LIMA_Y = np.where(VALID, FLOOD.astype(np.int8), np.int8(-1)).astype(np.int8)
save_geotiff(f"{LIMA_DIR}/etiqueta_EMSN038.tif", LIMA_Y)
print(f"Píxeles evaluables: {VALID.sum():,} | de ellos inundados según EMSN038: {(LIMA_Y == 1).sum():,} ({100*(LIMA_Y==1).sum()/max(VALID.sum(),1):.2f}%)")

### 10.2 Chips de 512×512 en formato Sen1Floods11

Para cumplir el formato pedido se recorta la escena y la etiqueta en chips de 512×512 (GeoTIFF, VV/VH en dB). Se guardan solo los chips que contienen píxeles evaluables.

In [ ]:
CHIP_DIR = f"{LIMA_DIR}/chips"; os.makedirs(CHIP_DIR, exist_ok=True)
lima_chips = []
for r0 in range(0, H - 511, 512):
    for c0 in range(0, W - 511, 512):
        y = LIMA_Y[r0:r0 + 512, c0:c0 + 512]
        if (y >= 0).sum() < 1000: continue
        tf = LIMA_TF * Affine.translation(c0, r0); name = f"Lima_r{r0//512:02d}_c{c0//512:02d}"
        for suf, arr, dt in [("S1Hand", LIMA["2017-03-17"][:, r0:r0 + 512, c0:c0 + 512], "float32"), ("LabelHand", y[None].astype(np.int16), "int16")]:
            with rasterio.open(f"{CHIP_DIR}/{name}_{suf}.tif", "w", driver="GTiff", height=512, width=512, count=arr.shape[0],
                               dtype=dt, crs="EPSG:4326", transform=tf) as d: d.write(arr.astype(dt))
        lima_chips.append((name, r0, c0, int((y == 1).sum())))
print(f"{len(lima_chips)} chips de Lima guardados en {CHIP_DIR} | chips con inundación: {sum(c[3] > 0 for c in lima_chips)}")

## 11. Inferencia *overlap-tile* sobre Lima y brecha de generalización

La escena completa (~5,500 × 4,000 px) no cabe de una vez en la GPU. Se usa la estrategia **overlap-tile** del paper (§2, Fig. 2; Clase 9 §B4): se predice por mosaicos de 512 px que se solapan, se conserva solo el **centro** de cada mosaico (donde la red ve contexto completo) y en los bordes de la escena el contexto faltante se rellena por **reflexión especular**.

El modelo se entrenó **solo** con eventos de Sen1Floods11 (ningún dato peruano).

In [ ]:
@torch.no_grad()
def predict_scene(model, x_db, tile=512, margin=64):
    # x_db: (2,H,W) en dB con NaN. Devuelve probabilidad de agua (H,W).
    x8, _ = db_to_u8(x_db); x = to_tensor(x8)
    Hs, Ws = x.shape[1:]; core = tile - 2 * margin
    nh, nw = math.ceil(Hs / core), math.ceil(Ws / core)
    pad = (margin, margin + nw * core - Ws, margin, margin + nh * core - Hs)        # izq, der, arriba, abajo
    xp = F.pad(x[None], pad, mode="reflect")[0]
    prob = torch.zeros(nh * core, nw * core)
    model.eval()
    for i in range(nh):
        for j in range(nw):
            t = xp[:, i * core:i * core + tile, j * core:j * core + tile][None].to(device)
            with torch.autocast(device.type, enabled=device.type == "cuda"):
                p = torch.sigmoid(model(t).float())[0].cpu()
            prob[i * core:(i + 1) * core, j * core:(j + 1) * core] = p[margin:margin + core, margin:margin + core]
    return prob[:Hs, :Ws].numpy()

PROB = {d: predict_scene(model, LIMA[d]) for d in LIMA}
for d, p in PROB.items(): save_geotiff(f"{LIMA_DIR}/prob_agua_{d}.tif", p.astype(np.float32))

m = WaterMetrics(); m.update(torch.from_numpy((PROB["2017-03-17"] > 0.5)[None].astype(np.int64)), torch.from_numpy(LIMA_Y[None]))
lima_res = m.summary()
# IoU promedio sobre los chips de Lima que tienen inundación (misma métrica de referencia que los autores)
chip_iou = []
for name, r0, c0, nf in lima_chips:
    if nf == 0: continue
    mm = WaterMetrics(); mm.update(torch.from_numpy((PROB["2017-03-17"][r0:r0+512, c0:c0+512] > .5)[None].astype(np.int64)),
                                   torch.from_numpy(LIMA_Y[r0:r0+512, c0:c0+512][None])); chip_iou.append(mm.summary()["IoU"])
lima_res["IoU_chip_medio"] = float(np.mean(chip_iou)) if chip_iou else float("nan")

# Detección a nivel de polígono: % de polígonos EMSN038 con ≥ 25 % de su área detectada como agua
flood_parts = flood_gdf.explode(index_parts=False)
det = []
for geom in flood_parts.geometry:
    if geom is None or geom.is_empty: continue
    w_, s_, e_, n_ = geom.bounds; c0, r0 = ~LIMA_TF * (w_, n_); c1, r1 = ~LIMA_TF * (e_, s_)
    r0, c0 = max(int(r0) - 1, 0), max(int(c0) - 1, 0); r1, c1 = min(int(r1) + 2, H), min(int(c1) + 2, W)
    if r1 <= r0 or c1 <= c0: continue
    mk = rasterize([(geom, 1)], out_shape=(r1 - r0, c1 - c0), transform=LIMA_TF * Affine.translation(c0, r0),
                   fill=0, dtype="uint8").astype(bool) & VALID[r0:r1, c0:c1]
    if mk.sum() >= 5: det.append((PROB["2017-03-17"][r0:r1, c0:c1][mk] > 0.5).mean())
lima_res["poligonos_detectados_25pct"] = float(np.mean(np.array(det) >= 0.25)) if det else float("nan")
lima_res["n_poligonos"] = len(det)

r_test = evaluate(model, *GOLD["test"][:2])
brecha = pd.DataFrame({"Prueba en distribución": r_test, "Lima 17-mar-2017 (EMSN038)": lima_res}).T
brecha = brecha[["IoU", "IoU_chip_medio", "F1", "precision", "recall", "omision", "comision"]]
brecha.loc["Brecha (Lima − dist.)"] = brecha.iloc[1] - brecha.iloc[0]
brecha.to_csv(f"{RES_DIR}/tabla_brecha_lima.csv"); print(brecha.round(3))
print(f"Polígonos EMSN038 evaluables: {lima_res['n_poligonos']} | detectados (≥25 % de su área): {100*lima_res['poligonos_detectados_25pct']:.0f}%")

# Recesión del agua: superficie detectada dentro de los polígonos EMSN038 el 17 vs. el 29 de marzo
inside = (LIMA_Y == 1)
extra = {"n_poligonos": lima_res["n_poligonos"], "pct_poligonos_detectados": 100 * lima_res["poligonos_detectados_25pct"],
         "km2_inundados_emsn": float((LIMA_Y == 1).sum() * (PIX * 111320) ** 2 / 1e6), "km2_evaluables": float(VALID.sum() * (PIX * 111320) ** 2 / 1e6)}
for d in ["2017-03-17", "2017-03-29"]:
    a_in, a_out = 100 * (PROB[d][inside] > .5).mean(), 100 * (PROB[d][(LIMA_Y == 0)] > .5).mean()
    extra[f"pct_agua_dentro_{d[-2:]}"], extra[f"pct_agua_fuera_{d[-2:]}"] = float(a_in), float(a_out)
    print(f"{d}: agua detectada dentro de los polígonos EMSN038 = {a_in:.1f}% | agua detectada en el resto del área evaluable = {a_out:.2f}%")
json.dump(extra, open(f"{RES_DIR}/lima_extra.json", "w"), indent=1)

### 11.1 Lima con todas las corridas de la ablación
¿La decisión que gana en validación (en distribución) es también la que mejor generaliza? Se evalúan las 7 corridas sobre Lima y se comparan con su IoU en prueba y en Bolivia.

In [ ]:
filas = {}
for name in load_results():
    mdl = load_model(name); pr = predict_scene(mdl, LIMA["2017-03-17"])
    mm = WaterMetrics(); mm.update(torch.from_numpy((pr > .5)[None].astype(np.int64)), torch.from_numpy(LIMA_Y[None]))
    r = load_results()[name]; L = mm.summary()
    filas[name] = {"IoU val": r["val"]["IoU"], "IoU test": r["test"]["IoU"], "IoU Bolivia": (r["bolivia"] or {}).get("IoU", np.nan),
                   "IoU Lima": L["IoU"], "Precisión Lima": L["precision"], "Recall Lima": L["recall"]}
    del mdl; torch.cuda.empty_cache()
tabla_lima_todas = pd.DataFrame(filas).T
tabla_lima_todas.to_csv(f"{RES_DIR}/tabla_lima_todas.csv"); print(tabla_lima_todas.round(3))

### 11.2 Diagnóstico: ¿el SAR del 17 de marzo "ve" agua dentro de los polígonos EMSN038?
Antes de culpar al modelo, se comprueba con la física: se compara la retrodispersión (dB) dentro de los polígonos EMSN038 con la del resto del área y con la del **agua** de Sen1Floods11. Además, se evalúa un umbral de Otsu calculado sobre la propia escena de Lima (línea base sin entrenamiento) y el **cambio** de retrodispersión entre el 17 y el 29 de marzo (misma órbita).

In [ ]:
from skimage.filters import threshold_otsu
diag = {}
inside, outside = (LIMA_Y == 1), (LIMA_Y == 0)
X_s1f, Y_s1f, _ = GOLD["train"]
vh_s1f_agua = u8_to_db(X_s1f[:, 1][Y_s1f == 1]); vh_s1f_seco = u8_to_db(X_s1f[:, 1][Y_s1f == 0])
filas = {}
for d in ["2017-03-17", "2017-03-29"]:
    for b, nb_ in enumerate(["VV", "VH"]):
        filas[(d, nb_)] = {"EMSN038 inundado": np.nanmedian(LIMA[d][b][inside]), "resto evaluable": np.nanmedian(LIMA[d][b][outside])}
filas[("Sen1Floods11", "VH")] = {"EMSN038 inundado": float(np.median(vh_s1f_agua)), "resto evaluable": float(np.median(vh_s1f_seco))}
tabla_db = pd.DataFrame(filas).T.rename(columns={"EMSN038 inundado": "mediana dB: agua/inundado", "resto evaluable": "mediana dB: no agua/resto"})
print(tabla_db.round(2)); tabla_db.to_csv(f"{RES_DIR}/tabla_lima_dB.csv")

# Línea base: umbral de Otsu sobre VH de la propia escena (solo píxeles evaluables)
vh17 = LIMA["2017-03-17"][1]; vals = vh17[VALID & ~np.isnan(vh17)]
thr = threshold_otsu(np.clip(vals, -35, 0)); diag["umbral_otsu_lima_VH_dB"] = float(thr)
mm = WaterMetrics(); mm.update(torch.from_numpy((vh17 < thr)[None].astype(np.int64)), torch.from_numpy(LIMA_Y[None])); o = mm.summary()
diag.update({f"otsu_{k}": float(v) for k, v in o.items() if k in ("IoU", "precision", "recall")})
print(f"\nOtsu sobre la escena de Lima: umbral VH = {thr:.1f} dB | IoU = {o['IoU']:.3f} | precisión = {o['precision']:.3f} | recall = {o['recall']:.3f}")

# Cambio 17→29 de marzo (misma órbita 171): el agua que se retira AUMENTA la retrodispersión
dVH = LIMA["2017-03-29"][1] - LIMA["2017-03-17"][1]
diag["cambio_VH_17a29_dentro_dB"] = float(np.nanmedian(dVH[inside])); diag["cambio_VH_17a29_fuera_dB"] = float(np.nanmedian(dVH[outside]))
diag["pct_dentro_VH_bajo_-20dB"] = float(100 * np.nanmean(vh17[inside] < -20)); diag["pct_s1f_agua_VH_bajo_-20dB"] = float(100 * np.mean(vh_s1f_agua < -20))
print(f"Cambio mediano de VH (29 − 17 mar): dentro de polígonos {diag['cambio_VH_17a29_dentro_dB']:+.2f} dB | fuera {diag['cambio_VH_17a29_fuera_dB']:+.2f} dB")
print(f"% de píxeles con VH < −20 dB: EMSN038 inundado (17-mar) {diag['pct_dentro_VH_bajo_-20dB']:.1f}% | agua Sen1Floods11 {diag['pct_s1f_agua_VH_bajo_-20dB']:.1f}%")
json.dump(diag, open(f"{RES_DIR}/lima_diagnostico.json", "w"), indent=1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
bins = np.linspace(-35, 0, 71)
ax[0].hist(vh_s1f_agua, bins, density=True, alpha=.5, color="#1f6fd1", label="Sen1Floods11: agua")
ax[0].hist(vh_s1f_seco, bins, density=True, alpha=.4, color="#8c8c8c", label="Sen1Floods11: no agua")
ax[0].hist(vh17[inside & ~np.isnan(vh17)], bins, density=True, histtype="step", lw=2, color="#d7301f", label="Lima 17-mar: dentro de EMSN038")
ax[0].hist(vh17[outside & ~np.isnan(vh17)], bins, density=True, histtype="step", lw=2, color="#333333", label="Lima 17-mar: resto")
ax[0].axvline(thr, ls="--", color="k", lw=1); ax[0].set(title="Distribución de VH (dB)", xlabel="VH (dB)"); ax[0].legend(fontsize=8)
ax[1].hist(dVH[inside & ~np.isnan(dVH)], np.linspace(-8, 8, 65), density=True, alpha=.6, color="#d7301f", label="dentro de EMSN038")
ax[1].hist(dVH[outside & ~np.isnan(dVH)], np.linspace(-8, 8, 65), density=True, histtype="step", lw=2, color="#333", label="resto")
ax[1].set(title="Cambio de VH entre el 17 y el 29 de marzo (dB)", xlabel="VH(29) − VH(17)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig_lima_histogramas.png", dpi=130, bbox_inches="tight"); plt.show()

## 12. Figuras de Lima

Dos zonas de detalle: **Huachipa–Carapongo–Cajamarquilla** (río Rímac / Huaycoloro) y **Comas–Carabayllo–Puente Piedra** (río Chillón). Contorno amarillo = polígonos EMSN038.

In [ ]:
ZONAS = {"Huachipa–Carapongo (Rímac/Huaycoloro)": (-76.96, -12.03, -76.84, -11.93),
         "Comas–Carabayllo (Chillón)":           (-77.12, -11.95, -76.99, -11.82)}

def win(bbox):
    w, s, e, n = bbox; c0, r0 = ~LIMA_TF * (w, n); c1, r1 = ~LIMA_TF * (e, s)
    return slice(int(max(r0, 0)), int(min(r1, H))), slice(int(max(c0, 0)), int(min(c1, W)))

for zona, bbox in ZONAS.items():
    rs, cs = win(bbox)
    if rs.stop - rs.start < 10 or cs.stop - cs.start < 10: continue
    ext = [bbox[0], bbox[2], bbox[1], bbox[3]]
    fig, ax = plt.subplots(1, 4, figsize=(20, 5.3))
    ax[0].imshow(LIMA["2017-03-17"][1][rs, cs], cmap="gray", vmin=-30, vmax=-5, extent=ext); ax[0].set_title("VH 17-mar-2017 (dB)")
    ax[1].imshow(PROB["2017-03-17"][rs, cs] > .5, cmap="Blues", extent=ext); ax[1].set_title("Predicción U-Net 17-mar")
    ax[2].imshow(error_map((PROB["2017-03-17"][rs, cs] > .5).astype(int), LIMA_Y[rs, cs]), cmap=ERR_CMAP, vmin=0, vmax=4,
                 interpolation="nearest", extent=ext); ax[2].set_title("Error vs EMSN038 (oscuro TP · rojo FP · azul FN)")
    ax[3].imshow(PROB["2017-03-29"][rs, cs] > .5, cmap="Blues", extent=ext); ax[3].set_title("Predicción U-Net 29-mar (recesión)")
    for a in ax:
        flood_gdf.boundary.plot(ax=a, color="#f5b800", lw=0.8); a.set_xlim(bbox[0], bbox[2]); a.set_ylim(bbox[1], bbox[3]); a.tick_params(labelsize=7)
    plt.suptitle(zona); plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/fig_lima_{zona.split('–')[0].lower()}.png", dpi=130, bbox_inches="tight"); plt.show()

## 13. Resumen para el informe

Tablas guardadas en `P2_inundaciones/resultados/` y figuras en `P2_inundaciones/figuras/`:

| Archivo | Sección del informe |
|---|---|
| `tabla_ablacion.csv`, `fig_curvas.png` | Experimentos — ablación (Entregable 2) |
| `tabla_evaluacion.csv`, `fig_matriz_confusion.png` | Resultados en distribución + Otsu (Entregable 3) |
| `tabla_brecha_lima.csv`, `fig_lima_*.png` | Generalización al Perú (Entregable 4) |
| `fig_errores_FP.png`, `fig_errores_FN.png` | Análisis cualitativo de fallos (Entregable 5) |

**Hipótesis a contrastar en la discusión de la brecha (Entregable 4):**
1. **Desajuste temporal y de sensor de la etiqueta:** EMSN038 delinea sobre WorldView-2 del 18 y 21 de marzo (≈ 0.5 m, óptico); el SAR es del 17 de marzo a las 05:38. Los huaicos son eventos de horas: el agua cambia entre ambas fechas. La comparación 17 vs. 29 de marzo cuantifica la recesión.
2. **Cobertura urbana y doble rebote:** gran parte de los polígonos están en zonas urbanas (Huachipa, SMP, Comas), donde el agua entre edificios devuelve señal brillante → FN. Ningún evento de Sen1Floods11 es una ciudad costera desértica.
3. **Escala:** las inundaciones de Lima son franjas angostas (decenas de metros) junto a cauces; a 10 m son pocos píxeles, y un desfase de 1–2 píxeles en el borde reduce mucho el IoU (ver % de polígonos detectados como métrica complementaria).
4. **Lodo vs. agua clara:** los huaicos son flujos de barro con alta carga de sedimentos y superficie rugosa, y dejan depósitos de lodo húmedo, no láminas de agua tranquila como en las llanuras de Sen1Floods11.
5. **Geometría del radar:** órbita descendente con valles encajonados (Rímac): sombras de radar en laderas → FP, y *layover* en laderas frente al satélite.

In [ ]:
print("MODELO FINAL:", BEST, "| config:", load_results()[BEST]["cfg"])
for f in ["tabla_ablacion.csv", "tabla_evaluacion.csv", "tabla_brecha_lima.csv"]:
    p = f"{RES_DIR}/{f}"
    if os.path.exists(p): print(f"\n=== {f}\n", pd.read_csv(p, index_col=0).round(3).to_string())

## 14. Empaquetar resultados para el informe
Crea `resultados_informe.zip` (tablas + figuras, sin pesos) en la carpeta de Drive y, en Colab, lo descarga.

In [ ]:
import shutil
zip_base = os.path.join(DRIVE_DIR, "resultados_informe")
tmp = "/tmp/resultados_informe"; shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(RES_DIR, f"{tmp}/resultados"); shutil.copytree(FIG_DIR, f"{tmp}/figuras")
shutil.make_archive(zip_base, "zip", tmp)
print("Creado:", zip_base + ".zip")
if IN_COLAB:
    from google.colab import files; files.download(zip_base + ".zip")